# 🗺️ PathSnap — Training Notebook
### Map Matching with ML + HMM | PES University Hackathon

**This notebook:**
1. Loads all 80 training trips from the dataset
2. Performs EDA
3. Builds feature matrix
4. Trains 5 models with Group-aware CV
5. Runs Optuna Bayesian tuning on best model
6. Evaluates on student-test trips
7. Saves all models as `.joblib` files


In [ ]:
# Install extra packages not in Kaggle by default
!pip install -q lightgbm optuna folium networkx

In [ ]:
import os, math, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from pathlib import Path
import joblib
import networkx as nx

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score, log_loss, confusion_matrix, ConfusionMatrixDisplay
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

print('✅ All imports successful')

In [ ]:
# ─── Set data paths (adjust if your Kaggle dataset name differs) ───
KAGGLE_INPUT = Path('/kaggle/input')

# Find the dataset directory automatically
dataset_dirs = list(KAGGLE_INPUT.iterdir())
print('Available datasets:', [d.name for d in dataset_dirs])

# Replace 'pathsnap-student-materials' with your actual Kaggle dataset slug
DATA_ROOT = dataset_dirs[0]  # Adjust index if multiple datasets
print('Using dataset:', DATA_ROOT)

TRAINING_DIR = DATA_ROOT / 'training'
STUDENT_TEST_DIR = DATA_ROOT / 'student_test'
EVAL_TEST_DIR = DATA_ROOT / 'eval_test'

OUTPUT_DIR = Path('/kaggle/working/models')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Training trips:', len(list(TRAINING_DIR.iterdir())) if TRAINING_DIR.exists() else 'NOT FOUND')
print('Student test trips:', len(list(STUDENT_TEST_DIR.iterdir())) if STUDENT_TEST_DIR.exists() else 'NOT FOUND')

## 1. Feature Engineering Functions

In [ ]:
# ══════════════════════════════════════════
#  GEOMETRY HELPERS
# ══════════════════════════════════════════

def haversine(lon1, lat1, lon2, lat2):
    R = 6_371_000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def bearing(lon1, lat1, lon2, lat2):
    lat1, lat2 = math.radians(lat1), math.radians(lat2)
    dlon = math.radians(lon2 - lon1)
    x = math.sin(dlon) * math.cos(lat2)
    y = math.cos(lat1)*math.sin(lat2) - math.sin(lat1)*math.cos(lat2)*math.cos(dlon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def dist_to_segment_m(px, py, ax, ay, bx, by):
    mid_lat = (ay + by) / 2
    dpm_lat = 1 / 111_111
    dpm_lon = 1 / (111_111 * math.cos(math.radians(mid_lat)))
    px_m, py_m = px/dpm_lon, py/dpm_lat
    ax_m, ay_m = ax/dpm_lon, ay/dpm_lat
    bx_m, by_m = bx/dpm_lon, by/dpm_lat
    dx, dy = bx_m - ax_m, by_m - ay_m
    if dx==0 and dy==0:
        return math.hypot(px_m-ax_m, py_m-ay_m)
    t = max(0, min(1, ((px_m-ax_m)*dx + (py_m-ay_m)*dy) / (dx*dx + dy*dy)))
    return math.hypot(px_m - (ax_m+t*dx), py_m - (ay_m+t*dy))

print('✅ Geometry helpers ready')

In [ ]:
# ══════════════════════════════════════════
#  DATA LOADERS
# ══════════════════════════════════════════

def load_track(path):
    df = pd.read_csv(path, sep=r'\s+', header=None,
                     names=['longitude','latitude','timestamp_sec'])
    return df.sort_values('timestamp_sec').reset_index(drop=True)

def load_nodes(path):
    df = pd.read_csv(path, sep=r'\s+', header=None, names=['longitude','latitude'])
    df.index.name = 'node_id'
    return df.reset_index()

def load_arcs(path):
    df = pd.read_csv(path, sep=r'\s+', header=None, names=['from_node_id','to_node_id'])
    df.index.name = 'arc_id'
    return df.reset_index()

def load_route(path):
    with open(path) as f:
        return [int(l.strip()) for l in f if l.strip()]

def build_arc_geometry(nodes_df, arcs_df):
    nl = nodes_df.set_index('node_id')[['longitude','latitude']].to_dict('index')
    arcs = []
    for _, row in arcs_df.iterrows():
        a, b = nl[row['from_node_id']], nl[row['to_node_id']]
        length = haversine(a['longitude'],a['latitude'],b['longitude'],b['latitude'])
        brg = bearing(a['longitude'],a['latitude'],b['longitude'],b['latitude'])
        arcs.append({'arc_id':row['arc_id'],'from_node_id':row['from_node_id'],
                     'to_node_id':row['to_node_id'],
                     'lon_a':a['longitude'],'lat_a':a['latitude'],
                     'lon_b':b['longitude'],'lat_b':b['latitude'],
                     'arc_length_m':length,'arc_bearing':brg})
    return arcs

FEATURE_COLS = ['dist_to_arc_m','bearing_diff','cos_bearing_diff',
                'sin_bearing_diff','speed_mps','time_delta_sec',
                'arc_length_m','candidate_rank']

def compute_features(track_df, arc_geom_list, route_arc_ids=None,
                      radius_m=50, top_k=10):
    rows = []
    for i, gps in track_df.iterrows():
        if i == 0:
            speed, dt, gps_brg = 0.0, 0.0, 0.0
        else:
            prev = track_df.iloc[i-1]
            dt = gps['timestamp_sec'] - prev['timestamp_sec']
            d = haversine(prev['longitude'],prev['latitude'],
                          gps['longitude'],gps['latitude'])
            speed = d/dt if dt>0 else 0.0
            gps_brg = bearing(prev['longitude'],prev['latitude'],
                              gps['longitude'],gps['latitude'])

        candidates = []
        for arc in arc_geom_list:
            d = dist_to_segment_m(gps['longitude'],gps['latitude'],
                                   arc['lon_a'],arc['lat_a'],
                                   arc['lon_b'],arc['lat_b'])
            if d <= radius_m:
                candidates.append({**arc,'dist_to_arc_m':d})
        candidates.sort(key=lambda x: x['dist_to_arc_m'])
        candidates = candidates[:top_k]

        gt_arc = route_arc_ids[i] if (route_arc_ids and i < len(route_arc_ids)) else None

        for rank, cand in enumerate(candidates):
            bdiff = (gps_brg - cand['arc_bearing'] + 360) % 360
            if bdiff > 180: bdiff = 360 - bdiff
            rows.append({
                'fix_index': i, 'arc_id': cand['arc_id'],
                'dist_to_arc_m': cand['dist_to_arc_m'],
                'bearing_diff': bdiff,
                'cos_bearing_diff': math.cos(math.radians(bdiff)),
                'sin_bearing_diff': math.sin(math.radians(bdiff)),
                'speed_mps': speed, 'time_delta_sec': dt,
                'arc_length_m': cand['arc_length_m'],
                'candidate_rank': rank+1,
                'label': 1 if (gt_arc is not None and cand['arc_id']==gt_arc) else 0
            })
    return pd.DataFrame(rows)

print('✅ Data loaders ready')

## 2. EDA — Exploratory Data Analysis

In [ ]:
# ── Sample one trip for EDA ──
sample_trips = sorted(list(TRAINING_DIR.iterdir()))[:5]
print(f'Sampling {len(sample_trips)} trips for EDA')

eda_stats = []
for trip_dir in sample_trips:
    track_f = list(trip_dir.glob('*.track'))[0]
    nodes_f = list(trip_dir.glob('*.nodes'))[0]
    arcs_f  = list(trip_dir.glob('*.arcs'))[0]
    route_f = list(trip_dir.glob('*.route'))
    
    track = load_track(track_f)
    nodes = load_nodes(nodes_f)
    arcs  = load_arcs(arcs_f)
    route = load_route(route_f[0]) if route_f else []
    
    intervals = track['timestamp_sec'].diff().dropna()
    gaps = (intervals > 5).sum()
    
    eda_stats.append({
        'trip': trip_dir.name,
        'n_fixes': len(track),
        'n_nodes': len(nodes),
        'n_arcs': len(arcs),
        'n_route_arcs': len(route),
        'mean_interval_sec': round(intervals.mean(), 2),
        'max_interval_sec': round(intervals.max(), 2),
        'n_gaps_gt5s': int(gaps),
        'duration_min': round((track['timestamp_sec'].max() - track['timestamp_sec'].min()) / 60, 1)
    })

eda_df = pd.DataFrame(eda_stats)
print('\n=== EDA Summary ===')
display(eda_df)

In [ ]:
# ── Sampling interval distribution ──
all_intervals = []
for trip_dir in sample_trips:
    track_f = list(trip_dir.glob('*.track'))[0]
    track = load_track(track_f)
    all_intervals.extend(track['timestamp_sec'].diff().dropna().tolist())

plt.figure(figsize=(10, 4))
plt.subplot(1,2,1)
plt.hist([x for x in all_intervals if x <= 10], bins=30, color='steelblue', edgecolor='white')
plt.xlabel('GPS Sampling Interval (sec)'); plt.ylabel('Count')
plt.title('GPS Sampling Interval Distribution')
plt.axvline(1.0, color='red', linestyle='--', label='1-sec expected')
plt.legend()

plt.subplot(1,2,2)
plt.hist(eda_df['n_arcs'], bins=10, color='coral', edgecolor='white')
plt.xlabel('Number of Arcs'); plt.ylabel('Count')
plt.title('Road Network Size per Trip')
plt.tight_layout()
plt.savefig('/kaggle/working/eda_intervals.png', dpi=150)
plt.show()
print('EDA plots saved.')

In [ ]:
# ── Folium map of one sample trip ──
import folium

sample = sample_trips[0]
track = load_track(list(sample.glob('*.track'))[0])
nodes = load_nodes(list(sample.glob('*.nodes'))[0])
arcs  = load_arcs(list(sample.glob('*.arcs'))[0])
route_f = list(sample.glob('*.route'))
route = load_route(route_f[0]) if route_f else []
arc_geom = build_arc_geometry(nodes, arcs)
arc_lookup = {a['arc_id']: a for a in arc_geom}

center_lat = track['latitude'].mean()
center_lon = track['longitude'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles='CartoDB positron')

# GPS track
folium.PolyLine([[r['latitude'],r['longitude']] for _,r in track.iterrows()],
                color='blue', weight=2, opacity=0.6, tooltip='GPS Track').add_to(m)

# Ground truth route
if route:
    gt_coords = []
    for arc_id in route:
        if arc_id in arc_lookup:
            a = arc_lookup[arc_id]
            gt_coords.append([a['lat_a'], a['lon_a']])
            gt_coords.append([a['lat_b'], a['lon_b']])
    if gt_coords:
        folium.PolyLine(gt_coords, color='red', weight=4, opacity=0.8,
                        tooltip='Ground Truth Route').add_to(m)

# Road network (grey)
for arc in arc_geom:
    folium.PolyLine([[arc['lat_a'],arc['lon_a']],[arc['lat_b'],arc['lon_b']]],
                    color='grey', weight=1, opacity=0.3).add_to(m)

m.save('/kaggle/working/sample_trip_map.html')
print('✅ Folium map saved to /kaggle/working/sample_trip_map.html')
m

## 3. Build Full Training Dataset

In [ ]:
print('🔄 Building training dataset from all 80 trips...')
print('(This may take 5–10 minutes on CPU; faster with Kaggle GPU)')

all_rows = []
trip_dirs = sorted(list(TRAINING_DIR.iterdir()))

for trip_id, trip_dir in enumerate(trip_dirs):
    track_f = list(trip_dir.glob('*.track'))
    nodes_f = list(trip_dir.glob('*.nodes'))
    arcs_f  = list(trip_dir.glob('*.arcs'))
    route_f = list(trip_dir.glob('*.route'))

    if not (track_f and nodes_f and arcs_f and route_f):
        print(f'  ⚠ Skipping {trip_dir.name} — missing files')
        continue

    track = load_track(track_f[0])
    nodes = load_nodes(nodes_f[0])
    arcs  = load_arcs(arcs_f[0])
    route = load_route(route_f[0])
    arc_geom = build_arc_geometry(nodes, arcs)

    feat_df = compute_features(track, arc_geom, route)
    feat_df['trip_id'] = trip_id
    all_rows.append(feat_df)

    if (trip_id + 1) % 10 == 0:
        print(f'  Progress: {trip_id+1}/{len(trip_dirs)} trips')

full_df = pd.concat(all_rows, ignore_index=True)
X = full_df[FEATURE_COLS]
y = full_df['label']
groups = full_df['trip_id']

print(f'\n✅ Dataset built!')
print(f'  Total rows: {len(full_df):,}')
print(f'  Positive (correct arc): {y.sum():,} ({y.mean()*100:.1f}%)')
print(f'  Negative (wrong arc):   {(1-y).sum():,}')
print(f'  Features: {list(FEATURE_COLS)}')

In [ ]:
# ── Feature distributions ──
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flatten(), FEATURE_COLS):
    full_df[full_df['label']==0][col].hist(bins=40, alpha=0.6, ax=ax, label='Wrong arc', color='salmon')
    full_df[full_df['label']==1][col].hist(bins=40, alpha=0.6, ax=ax, label='Correct arc', color='steelblue')
    ax.set_title(col); ax.legend(fontsize=7)
plt.suptitle('Feature Distributions: Correct vs Wrong Arc Candidates')
plt.tight_layout()
plt.savefig('/kaggle/working/feature_distributions.png', dpi=150)
plt.show()

## 4. Model Training with Group-Aware CV

In [ ]:
GKF = GroupKFold(n_splits=5)

def train_and_evaluate(name, model, X, y, groups, needs_scaling=False):
    print(f'\n🔄 Training: {name}')
    t_start = time.time()

    if needs_scaling:
        pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])
    else:
        pipeline = Pipeline([('model', model)])

    cv_scores = []
    for fold, (train_idx, val_idx) in enumerate(GKF.split(X, y, groups)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        pipeline.fit(X_tr, y_tr)
        y_pred = pipeline.predict(X_val)
        score = f1_score(y_val, y_pred, average='macro', zero_division=0)
        cv_scores.append(score)
        print(f'  Fold {fold+1}: Macro F1 = {score:.4f}')

    # Fit final model on all training data
    pipeline.fit(X, y)
    t_elapsed = time.time() - t_start

    result = {
        'name': name,
        'cv_macro_f1_mean': np.mean(cv_scores),
        'cv_macro_f1_std': np.std(cv_scores),
        'training_time_sec': round(t_elapsed, 1),
        'pipeline': pipeline
    }

    model_path = OUTPUT_DIR / f'{name.lower().replace(" ","_")}.joblib'
    joblib.dump(pipeline, model_path)
    print(f'  ✅ CV Macro F1: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}')
    print(f'  💾 Saved to {model_path}')
    return result

print('✅ Training function ready. Starting model training...')

In [ ]:
# ══════════════════════════════════════════
#  MODEL 1 — Logistic Regression (Baseline)
# ══════════════════════════════════════════
results = []

r = train_and_evaluate(
    'logistic_regression',
    LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs'),
    X, y, groups, needs_scaling=True
)
results.append(r)

In [ ]:
# ══════════════════════════════════════════
#  MODEL 2 — k-NN
# ══════════════════════════════════════════
r = train_and_evaluate(
    'knn',
    KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1),
    X, y, groups, needs_scaling=True
)
results.append(r)

In [ ]:
# ══════════════════════════════════════════
#  MODEL 3 — Random Forest
# ══════════════════════════════════════════
r = train_and_evaluate(
    'random_forest',
    RandomForestClassifier(n_estimators=200, max_depth=15,
                           min_samples_leaf=5, n_jobs=-1, random_state=42),
    X, y, groups, needs_scaling=False
)
results.append(r)

In [ ]:
# ══════════════════════════════════════════
#  MODEL 4 — LightGBM (default params)
# ══════════════════════════════════════════
r = train_and_evaluate(
    'lgbm',
    lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                       num_leaves=63, min_child_samples=20,
                       subsample=0.8, colsample_bytree=0.8,
                       n_jobs=-1, random_state=42, verbose=-1),
    X, y, groups, needs_scaling=False
)
results.append(r)

In [ ]:
# ══════════════════════════════════════════
#  MODEL 5 — MLP Neural Network
# ══════════════════════════════════════════
r = train_and_evaluate(
    'mlp',
    MLPClassifier(hidden_layer_sizes=(256, 128, 64),
                  activation='relu', solver='adam',
                  max_iter=200, early_stopping=True,
                  random_state=42),
    X, y, groups, needs_scaling=True
)
results.append(r)

In [ ]:
# ── Results Table ──
results_df = pd.DataFrame([{
    'Model': r['name'],
    'CV Macro F1': f"{r['cv_macro_f1_mean']:.4f} ± {r['cv_macro_f1_std']:.4f}",
    'Training Time (s)': r['training_time_sec']
} for r in results])
print('\n=== Model Comparison ===')
display(results_df)

best_result = max(results, key=lambda x: x['cv_macro_f1_mean'])
print(f'\n🏆 Best model (CV): {best_result["name"]} '
      f'(F1={best_result["cv_macro_f1_mean"]:.4f})')

## 5. Bayesian Hyperparameter Tuning with Optuna (LightGBM)

In [ ]:
print('🔄 Starting Optuna Bayesian optimization for LightGBM...')
print('(50 trials — takes ~15-25 mins on CPU, faster on GPU)')

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'n_jobs': -1, 'random_state': 42, 'verbose': -1
    }
    model = lgb.LGBMClassifier(**params)
    # Use first 3 folds only for speed
    scores = []
    for train_idx, val_idx in list(GKF.split(X, y, groups))[:3]:
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        y_pred = model.predict(X.iloc[val_idx])
        scores.append(f1_score(y.iloc[val_idx], y_pred, average='macro', zero_division=0))
    return np.mean(scores)

study = optuna.create_study(direction='maximize', study_name='lgbm_pathsnap')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\n✅ Best trial: {study.best_trial.number}')
print(f'   Best CV Macro F1: {study.best_trial.value:.4f}')
print(f'   Best params: {study.best_trial.params}')

In [ ]:
# ── Train final tuned LightGBM ──
best_params = {**study.best_trial.params, 'n_jobs': -1, 'random_state': 42, 'verbose': -1}
tuned_lgbm = lgb.LGBMClassifier(**best_params)

r_tuned = train_and_evaluate(
    'lgbm_tuned',
    tuned_lgbm,
    X, y, groups, needs_scaling=False
)

print(f'\n📊 Tuning comparison:')
print(f'   LightGBM (default): {[r for r in results if r["name"]=="lgbm"][0]["cv_macro_f1_mean"]:.4f}')
print(f'   LightGBM (tuned):   {r_tuned["cv_macro_f1_mean"]:.4f}')

In [ ]:
# ── Optuna visualization ──
try:
    from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
    fig = plot_optimization_history(study)
    plt.title('Optuna: Optimization History')
    plt.savefig('/kaggle/working/optuna_history.png', dpi=150)
    plt.show()

    fig2 = plot_param_importances(study)
    plt.title('Optuna: Hyperparameter Importance')
    plt.savefig('/kaggle/working/optuna_importance.png', dpi=150)
    plt.show()
except Exception as e:
    print(f'Optuna viz skipped: {e}')

## 6. Per-Model Analysis (Learning Curves + Confusion Matrices)

In [ ]:
# ── Confusion matrices on a held-out fold ──
train_idx, val_idx = list(GKF.split(X, y, groups))[0]
X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

model_names = ['logistic_regression','knn','random_forest','lgbm','mlp']

fig, axes = plt.subplots(1, len(model_names), figsize=(20, 4))
for ax, mname in zip(axes, model_names):
    model_path = OUTPUT_DIR / f'{mname}.joblib'
    if not model_path.exists():
        ax.set_title(f'{mname}\n(not found)'); continue
    pipe = joblib.load(model_path)
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Wrong','Correct']).plot(ax=ax)
    ax.set_title(mname.replace('_',' ').title())

plt.suptitle('Confusion Matrices (Fold 1)', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature importance (Random Forest + LightGBM) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, mname in zip(axes, ['random_forest', 'lgbm_tuned']):
    model_path = OUTPUT_DIR / f'{mname}.joblib'
    if not model_path.exists():
        model_path = OUTPUT_DIR / f'{mname.replace("_tuned","")}.joblib'
    if not model_path.exists():
        continue
    pipe = joblib.load(model_path)
    clf = pipe.named_steps.get('model') or pipe[-1]
    importances = clf.feature_importances_
    idx = np.argsort(importances)[::-1]
    ax.barh([FEATURE_COLS[i] for i in idx], importances[idx], color='steelblue')
    ax.set_xlabel('Importance'); ax.set_title(mname.replace('_',' ').title())

plt.suptitle('Feature Importances')
plt.tight_layout()
plt.savefig('/kaggle/working/feature_importance.png', dpi=150)
plt.show()

## 7. Final Summary & Model Export

In [ ]:
print('\n' + '='*60)
print('           PATHSNAP — FINAL MODEL SUMMARY')
print('='*60)

all_results = results + [r_tuned]
summary = pd.DataFrame([{
    'Model': r['name'].replace('_',' ').title(),
    'CV Macro F1 (mean)': round(r['cv_macro_f1_mean'], 4),
    'CV Macro F1 (std)': round(r['cv_macro_f1_std'], 4),
    'Train Time (s)': r['training_time_sec'],
    'Saved As': r['name'] + '.joblib'
} for r in all_results]).sort_values('CV Macro F1 (mean)', ascending=False)

display(summary)

print('\n📂 Models saved to /kaggle/working/models/')
for f in sorted(OUTPUT_DIR.glob('*.joblib')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:40s} {size_kb:.1f} KB')

print('\n✅ Training complete! Download the models/ folder from Kaggle output.')
print('   Place all .joblib files in your local Hackathon/models/ directory.')